In [1]:
import numpy as np 
import pandas as pd 

In [5]:
from pathlib import Path
BASE = Path.cwd()    
BASE

WindowsPath('d:/Downloads')

In [ ]:
# This orignal dataset is fully synthetic, generated with NumPy (see gen.py) 

train = pd.read_csv("/data/train.csv")
test = pd.read_csv("/data/test.csv")
orignal = pd.read_csv("/data/credit_card_fraud_2026.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'data/train.csv'

In [ ]:
ID = 'id'
TARGET = 'is_fraud'
SEED = 42

In [ ]:
train.head()

,id,amount_usd,hours_since_last_txn,txn_count_last_24h,distance_from_home_km,card_age_months,customer_age,account_balance_usd,cvv_retry_count,velocity_score,...,auth_method,channel,device_type,is_foreign_transaction,is_new_merchant,used_vpn,ip_country_mismatch,billing_shipping_mismatch,is_ai_generated_scam_attempt,is_fraud
0,0,61.16,9.77,1,17.78,48,56,1269.35,0,5.5,...,OTP,POS,iPhone,False,False,False,False,False,False,0.0
1,1,24.83,1.54,3,13.06,38,28,3233.00,0,33.4,...,PIN,In-App,Android Phone,False,False,False,False,False,False,0.0
2,2,6.36,1.20,4,8.17,50,55,1046.88,0,35.8,...,OTP,Online,Tablet,False,False,True,False,False,False,0.0
3,3,37.44,5.13,4,8.67,48,48,2217.67,0,20.7,...,3D Secure,POS,Android Phone,False,False,False,False,False,False,0.0
4,4,653.85,0.80,4,11.69,37,52,3291.06,0,34.8,...,3D Secure,Online,Windows PC,False,False,False,False,False,False,0.0


In [ ]:
train['is_fraud'].value_counts()

is_fraud
0.0    688271
1.0     11689
Name: count, dtype: int64

In [ ]:
orignal

,transaction_id,amount_usd,merchant_category,card_type,auth_method,channel,device_type,is_foreign_transaction,hours_since_last_txn,txn_count_last_24h,...,ip_country_mismatch,billing_shipping_mismatch,cvv_retry_count,velocity_score,time_of_day_hour,day_of_week,is_ai_generated_scam_attempt,merchant_risk_score,prior_disputes,is_fraud
0,1,42.86,Restaurants,Visa,OTP,Online,Android Phone,False,13.54,2,...,False,False,0,0.1,18,3,False,42.3,0,0
1,2,4.75,Online Retail,Mastercard,3D Secure,Online,Android Phone,False,0.71,2,...,False,False,0,25.8,12,4,False,28.3,0,0
2,3,77.18,Groceries,Mastercard,3D Secure,Online,Mac,False,0.35,5,...,False,True,0,42.3,5,0,False,24.7,1,0
3,4,1.69,Streaming,Visa,No Authentication,POS,Android Phone,False,3.42,6,...,False,False,0,28.9,22,6,False,56.2,1,0
4,5,261.68,Travel,Visa,3D Secure,In-App,iPhone,False,2.43,2,...,False,False,0,3.9,2,4,False,32.7,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,19996,41.40,Healthcare,Mastercard,OTP,POS,Android Phone,False,8.82,2,...,False,False,0,17.3,2,3,False,29.6,1,0
19996,19997,47.92,Online Retail,Visa,OTP,POS,Android Phone,False,45.81,1,...,False,False,0,9.0,4,0,False,29.2,1,0
19997,19998,10.25,Restaurants,Visa,3D Secure,In-App,Tablet,False,20.67,0,...,False,False,1,1.0,19,5,False,22.9,0,0
19998,19999,31.34,Travel,Visa,3D Secure,Online,Android Phone,False,22.94,2,...,False,False,0,8.8,2,1,False,27.9,0,0


In [ ]:
orignal['is_fraud'].value_counts()

is_fraud
0    19661
1      339
Name: count, dtype: int64

In [ ]:
train.shape

(699960, 26)

### 按照属性的类型分类

In [ ]:
FEATURE = [c for c in test.columns if c != ID]
CATEGORY = train[FEATURE].select_dtypes(include = 'object').columns.tolist()
NUMERICAL = train[FEATURE].select_dtypes(include = 'number').columns.tolist()
BOOL = train[FEATURE].select_dtypes(include = 'bool').columns.tolist()

print(f"FEATURE: {len(FEATURE)}")
print(f"CATEGORY: {len(CATEGORY)}")
print(f"NUMEROUS: {len(NUMERICAL)}")
print(f"BOOL: {len(BOOL)}")

FEATURE: 24
CATEGORY: 5
NUMEROUS: 13
BOOL: 6


### 数据集的大小

In [ ]:
print(f"The length of train:{len(train)}")
print(f"The length of test:{len(test)}")
print(f"The percent of is_fraud:{train[TARGET].value_counts(normalize = True)[1] * 100:.2}%" )
print(f"The length of:{len(orignal)}")

The length of train:699960
The length of test:299984
The percent of is_fraud:1.7%
The length of:20000


### 特征字典

In [ ]:
DESCR = {
    'amount_usd': 'Transaction amount, USD',
    'hours_since_last_txn': 'Hours since previous transaction',
    'txn_count_last_24h': 'Transactions in the last 24 hours',
    'distance_from_home_km': 'Distance from home, km',
    'card_age_months': 'Card age, months',
    'customer_age': 'Customer age, years',
    'account_balance_usd': 'Account balance, USD',
    'cvv_retry_count': 'Repeated CVV attempts',
    'velocity_score': 'Transaction velocity/intensity',
    'time_of_day_hour': 'Hour of day',
    'day_of_week': 'Day of week (0–6)',
    'merchant_risk_score': 'Merchant risk score',
    'prior_disputes': 'Prior disputed transactions',
    'merchant_category': 'Merchant category',
    'card_type': 'Card payment network',
    'auth_method': 'Authentication method',
    'channel': 'Transaction channel',
    'device_type': 'Device type',
    'is_foreign_transaction': 'Foreign transaction',
    'is_new_merchant': 'New merchant',
    'used_vpn': 'VPN usage',
    'ip_country_mismatch': 'IP country mismatch',
    'billing_shipping_mismatch': 'Billing/shipping address mismatch',
    'is_ai_generated_scam_attempt': 'AI-generated scam attempt flag'
}

dictionary = pd.DataFrame({
    'Feature' : FEATURE ,
    'Group' : [ 'Numerical' if c in NUMERICAL else 'CATEGORY' if c in CATEGORY 
                else 'BOOLEAN' for c in FEATURE] ,
    'Description' : [DESCR[c] for c in FEATURE]
})

dictionary

,Feature,Group,Description
0,amount_usd,Numerical,"Transaction amount, USD"
1,hours_since_last_txn,Numerical,Hours since previous transaction
2,txn_count_last_24h,Numerical,Transactions in the last 24 hours
3,distance_from_home_km,Numerical,"Distance from home, km"
4,card_age_months,Numerical,"Card age, months"
5,customer_age,Numerical,"Customer age, years"
6,account_balance_usd,Numerical,"Account balance, USD"
7,cvv_retry_count,Numerical,Repeated CVV attempts
8,velocity_score,Numerical,Transaction velocity/intensity
9,time_of_day_hour,Numerical,Hour of day


### 数据质量检查

In [ ]:
audit = pd.DataFrame(index = FEATURE)
audit['Type'] = train[FEATURE].dtypes.astype(str)
audit['Unique in test'] = test[FEATURE].nunique(dropna = False)
audit['Unique in train'] = train[FEATURE].nunique(dropna = False)
audit['Missing in test'] = test[FEATURE].isna().sum()
audit['Missing in train'] = train[FEATURE].isna().sum()

audit

,Type,Unique in test,Unique in train,Missing in test,Missing in train
amount_usd,float64,10510,13387,0,0
hours_since_last_txn,float64,3331,3633,0,0
txn_count_last_24h,int64,11,12,0,0
distance_from_home_km,float64,4108,4365,0,0
card_age_months,int64,49,49,0,0
customer_age,int64,64,64,0,0
account_balance_usd,float64,31803,43317,0,0
cvv_retry_count,int64,4,4,0,0
velocity_score,float64,644,666,0,0
time_of_day_hour,int64,24,24,0,0
